---
title: "2.1 Loading data and looking at it"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/01-loading-data.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/02-data/01-loading-data.ipynb){.btn .btn-outline-secondary}
:::

**No install required.** To run and save in Colab, sign in with a Google account. If the GitHub link cannot open, download the notebook, go to [Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and select the downloaded `.ipynb` file. A notebook is a document containing text and runnable code.

**Time:** about 35–40 minutes · **Before you begin:** complete [1.6 Importing libraries](../01-basics/06-importing-libraries.ipynb). This lesson loads a fresh table and does not depend on variables from another notebook.

## You will learn

- Load a CSV file into a pandas DataFrame.
- Check the table's size, column names, data types and first rows.
- Read the same observations from Excel and Stata files.
- Diagnose a missing file or a misspelled column name.

## Why it matters

Before estimating a regression, check that you loaded the data you meant to load. A table can open successfully and still have the wrong columns, years or units. We practise with made-up numbers for fictional countries, then inspect the real country panel behind the income–life expectancy question.

## Worked example

### 1. Meet a table small enough to check by hand

A **library** is a collection of reusable code. `import pandas as pd` loads the pandas library and gives it the short name `pd`. A **DataFrame** is pandas' table: columns hold variables and rows hold observations. Here each observation is one country in one year.

This cell makes a two-row table. Each column name is followed by a list of values in square brackets. Braces group the named columns; quotation marks mark text. The numbers below are made-up numbers.

In [ ]:
import pandas as pd

small = pd.DataFrame({
    "country": ["Northland", "Eastria"],
    "year": [2007, 2007],
    "life_exp": [72.0, 78.0],
    "gdp_pc": [12000, 28000],
})
small

Notice two rows and four named columns. The extra numbers at the far left are the **index**, pandas' row labels; they are not another country variable.

### 2. Get the real course file

A **CSV** is a text file of comma-separated values: the first line names the columns, and the remaining lines hold rows. Our committed CSV is a renamed copy of Jennifer Bryan's CC0 Gapminder extract: 142 countries, 1952–2007 at five-year intervals. It is a **panel**, meaning the same countries are observed repeatedly over time.

Run the prepared setup below. `Path` describes a file location. `data/gapminder.csv` means a file called `gapminder.csv` inside a folder called `data`. The setup uses that file when available, or downloads a fixed public extract and creates it in a fresh Colab runtime. It checks the download before using it. You can run the setup without learning all of its commands yet.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. This downloads no private data and does not connect to your Google Drive. Colab files disappear when the runtime is reset, so rerun the setup after a reset.

### 3. Read and inspect

`pd.read_csv(...)` reads a CSV into a DataFrame. Store it as `countries`, then use `.head()` to show its first five rows. The dot asks a table to perform an operation; the parentheses call that operation.

In [ ]:
countries = pd.read_csv(data_file)
countries.head()

Notice Afghanistan in several years: a row is a **country-year**, not a unique country. `gdp_pc` is GDP per person in inflation-adjusted US dollars as documented by the upstream extract; `life_exp` is life expectancy at birth in years. Neither describes a particular household. `region` uses the extract's broad continent labels. [Full provenance and units](https://github.com/ningzhanglab/Python-for-Beginners/blob/main/data/README.md).

`.shape` reports the number of rows followed by the number of columns. `.columns` lists the exact column names. These are properties, so they have no parentheses.

In [ ]:
print(countries.shape)
print(countries.columns.tolist())

Expect `(1704, 6)` and the names `country`, `region`, `year`, `life_exp`, `population`, `gdp_pc`. A **list** is an ordered collection; `.tolist()` makes the column names easy to read. This count is 142 countries × 12 years, not 1,704 different countries.

`.info()` reports column **data types** (kinds of values, such as text, whole numbers or decimal numbers) and non-missing counts. A **missing value** means a measurement was not recorded. A non-missing count of 1,704 in every column means this teaching extract has no gaps in those columns.

In [ ]:
countries.info()

Read `year` as a whole number, `life_exp` and `gdp_pc` as decimals, and `country` and `region` as text. Depending on your pandas version, text may be labelled `str`, `string` or `object`. A numeric-looking column stored as text needs investigation; do not assume calculations will work.

To check coverage, select the `year` column with square brackets and ask for its smallest and largest values.

In [ ]:
print(countries["year"].min())
print(countries["year"].max())

The years run from 1952 to 2007. These are historical teaching data, not current country estimates. Always check the source, measurement definitions and coverage before interpreting results.

### 4. Read Excel and Stata files

**Excel** workbooks usually end in `.xlsx`; **Stata** data files end in `.dta`. Changing a file's name does not change its format. For this demonstration, we write the same country observations to both formats in a disposable `scratch` folder. These are format conversions, not extra datasets or new sources.

Run this preparation cell to create the two practice files. `index=False` avoids saving pandas' row labels as an extra column. `sheet_name` names the Excel worksheet. Excel support uses the `openpyxl` library. Colab normally includes it; if it is missing, follow the course environment instructions or ask for help.

In [ ]:
practice_folder = Path("scratch")
practice_folder.mkdir(exist_ok=True)
countries.to_excel(practice_folder / "countries.xlsx", sheet_name="countries", index=False)
countries.to_stata(practice_folder / "countries.dta", write_index=False, version=118)
print("Created scratch/countries.xlsx and scratch/countries.dta")

Notice the file locations. The `/` joins folder and file names when using `Path`. The original course CSV is unchanged.

Use `read_excel` for Excel. Choose the worksheet explicitly to avoid silently loading a cover page or notes sheet.

In [ ]:
excel_countries = pd.read_excel(
    practice_folder / "countries.xlsx", sheet_name="countries", engine="openpyxl"
)
excel_countries.head(3)

The first three rows should match the CSV. This course pins the Excel-reading dependency in `requirements.txt`; no Excel purchase or local Excel installation is needed.

Use `read_stata` for Stata. `convert_categoricals=False` keeps stored codes rather than replacing them with value labels. This practice file has no labelled categories, so there is no difference here; for a real survey, inspect the codebook before choosing.

In [ ]:
stata_countries = pd.read_stata(
    practice_folder / "countries.dta", convert_categoricals=False
)
stata_countries.head(3)

The rows should again match. On a real file, check variable labels, date conversion and special missing values before proceeding.

Check that the conversions kept the number of rows and columns.

In [ ]:
print("CSV:", countries.shape)
print("Excel:", excel_countries.shape)
print("Stata:", stata_countries.shape)

All three should be `(1704, 6)`. Matching sizes alone do not guarantee matching values; look at the first rows and definitions too. The automated course checks also compare the converted values.

## Try it

Attempt each task in a new code cell before opening its solution.

### 1. Show more observations

Use `.head(8)` on `countries`. Why are the first eight rows not eight countries?

<details><summary>Solution 1</summary>

Ask for eight rows rather than the default five.

In [ ]:
countries.head(8)

They are different years for Afghanistan. The table is ordered by country and then year.

</details>

### 2. Count unique countries

`.nunique()` counts different values in a column. Use it on `countries["country"]`. Compare the result to the number of rows.

<details><summary>Solution 2</summary>

Select the country column and count distinct names.

In [ ]:
countries["country"].nunique()

There are 142 countries and 1,704 country-year rows. Repeated observations are part of a panel, not automatically duplicate mistakes.

</details>

### 3. Check the measurement range

Find the smallest and largest `life_exp` in the real panel using `.min()` and `.max()`. What are the units? Does the range alone establish that the data are correct?

<details><summary>Solution 3</summary>

Apply the two range checks to life expectancy.

In [ ]:
print(countries["life_exp"].min())
print(countries["life_exp"].max())

The range is 23.599 to 82.603 **years**. A plausible range is one useful check; it does not prove correct units, country coverage or source quality.

</details>

### 4. Check an imported format

Reload the Excel worksheet under the new name `reloaded`. Print its column names and its shape. Compare them with the CSV results.

<details><summary>Solution 4</summary>

Read the named worksheet and inspect its structure.

In [ ]:
reloaded = pd.read_excel(
    practice_folder / "countries.xlsx", sheet_name="countries", engine="openpyxl"
)
print(reloaded.columns.tolist())
print(reloaded.shape)

The six column names and 1,704 rows match. On your own project, retain the original raw file and document any conversion.

</details>

## When it goes wrong

### “FileNotFoundError”

Python looks for a relative file location inside the **working folder**, the folder the runtime currently uses. A **FileNotFoundError** means it did not find that location. This demonstration asks for a deliberately absent file. `try`/`except` catches the expected error to let the lesson continue; the wrapper is taught later.

In [ ]:
try:
    pd.read_csv("data/this-file-does-not-exist.csv")
except FileNotFoundError as error:
    print(f"{type(error).__name__}: {error}")

Check the folder, spelling and file ending. This next cell displays the working folder and whether the actual course file exists.

In [ ]:
print(Path.cwd())
print(data_file.exists())

The exact folder differs across computers; the second line should be `True`. If it is `False`, rerun the setup. For your own Colab files, use the folder icon on the left and its upload button; then use the uploaded file's location. Files uploaded to the runtime are temporary.

### “KeyError”

A **KeyError** means a requested label is absent. Python distinguishes capital letters. This deliberately asks for a column named `GDP`, then catches and displays the expected error.

In [ ]:
try:
    countries["GDP"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Inspect `.columns` to find the exact spelling. In this dataset the column is `gdp_pc`, so the corrected selection is:

In [ ]:
countries["gdp_pc"].head()

You should see the first five GDP-per-person values. If a CSV appears as one giant column, the separator may be different: some files use semicolons or tabs. Check the raw file before choosing a `sep` argument. Do not fix a format problem by guessing column names.

## Recap

- `pd.read_csv`, `pd.read_excel` and `pd.read_stata` read different formats.
- Inspect `.head()`, `.shape`, `.columns` and `.info()` before analysis.
- A row here is a country-year; units and coverage come from the source documentation.
- Keep raw files, document transformations and rerun from the top after a reset.

**Next:** [2.2 Selecting rows and columns](02-selecting.ipynb). Use conditions to define an analysis sample before transforming variables.

::: {.callout-note title="Coming from Stata / R / Excel?"}
Stata's `import delimited`, R's `read.csv` and Excel's text import all perform a role like `pd.read_csv`. pandas stores a named table in memory; several tables can coexist. You do not need to know any of these tools to take this course.
:::